# Lab 6: K-Means Clustering (Capstone)

---

## Overview

This capstone lab brings together all the GPU programming concepts you've learned to implement **K-Means clustering**, a classic unsupervised machine learning algorithm. K-Means demonstrates how real-world GPU applications combine multiple kernels and optimization patterns.

This lab synthesizes concepts from previous labs:
- **Lab 1 (Vector Addition)**: Thread indexing and kernel launch configuration
- **Lab 2 (Histogram)**: Atomic operations for parallel accumulation
- **Lab 3 (Reduction)**: Shared memory and two-level reduction
- **Lab 4 (Monte Carlo)**: Combining parallel computation with reduction

K-Means is widely used in data mining, image segmentation, customer segmentation, and as a preprocessing step for other machine learning algorithms.

## Learning Objectives

By the end of this lab, you will be able to:

1. Decompose an iterative algorithm into multiple GPU kernels
2. Implement the assignment step (parallel nearest-neighbor search)
3. Implement the update step using shared memory atomic reduction
4. Design convergence detection on GPU with early termination
5. Analyze the computational complexity of multi-kernel workflows

## 1. K-Means Algorithm

### Mathematical Formulation

**Assignment Step**: For each point $(x_i, y_i)$, find the nearest centroid:

$$
\text{label}_i = \arg\min_{0 \le j < k} \left( (x_i - \mu_{j,x})^2 + (y_i - \mu_{j,y})^2 \right)
$$

**Update Step**: Recompute each centroid as the mean of its assigned points:

$$
\mu_{j,x} = \frac{1}{|S_j|} \sum_{i \in S_j} x_i, \quad \mu_{j,y} = \frac{1}{|S_j|} \sum_{i \in S_j} y_i
$$

where $S_j$ is the set of points assigned to centroid $j$.

### Algorithm Flow

```
Initialize centroids
repeat:
    1. Assign each point to nearest centroid  [Parallel over points]
    2. Recompute centroids                    [Reduction + Division]
    3. Check convergence                      [Parallel comparison]
until converged or max_iterations reached
```

## 2. GPU Implementation Analysis

The implementation uses three kernels:

### Kernel 1: Find Nearest Centroids

```cpp
__global__ void find_nearest_centroids_kernel(
    const float* data_x, const float* data_y, int* labels,
    const float* centroids_x, const float* centroids_y,
    int sample_size, int k) 
{
    int point_id = blockIdx.x * blockDim.x + threadIdx.x;
    if (point_id < sample_size) {
        int nearest = 0;
        float min_dist = INFINITY;
        for (int j = 0; j < k; ++j) {
            float dx = centroids_x[j] - data_x[point_id];
            float dy = centroids_y[j] - data_y[point_id];
            float dist = dx*dx + dy*dy;
            if (dist < min_dist) {
                min_dist = dist;
                nearest = j;
            }
        }
        labels[point_id] = nearest;
    }
}
```

| Aspect | Analysis |
|:-------|:---------|
| Parallelism | One thread per data point |
| Work per thread | O(k) distance comparisons |
| Memory access | Coalesced reads of data, broadcast reads of centroids |

### Kernel 2: Recalculate Centroids

```cpp
__global__ void recalculate_centroid_kernel(
    const float* data_x, const float* data_y, const int* labels,
    int* centroid_point_nums, float* new_centroids_x, float* new_centroids_y,
    int sample_size, int k) 
{
    __shared__ float new_centroids_x_shared[100];
    __shared__ float new_centroids_y_shared[100];
    __shared__ int centroid_point_nums_shared[100];

    // Initialize shared memory
    if (threadIdx.x < k) {
        new_centroids_x_shared[threadIdx.x] = 0;
        new_centroids_y_shared[threadIdx.x] = 0;
        centroid_point_nums_shared[threadIdx.x] = 0;
    }
    __syncthreads();

    // Accumulate to shared memory (local atomics)
    int point_id = blockIdx.x * blockDim.x + threadIdx.x;
    if (point_id < sample_size) {
        int cid = labels[point_id];
        atomicAdd(new_centroids_x_shared + cid, data_x[point_id]);
        atomicAdd(new_centroids_y_shared + cid, data_y[point_id]);
        atomicAdd(centroid_point_nums_shared + cid, 1);
    }
    __syncthreads();

    // Merge to global memory (one atomic per block)
    if (threadIdx.x < k) {
        atomicAdd(new_centroids_x + threadIdx.x, new_centroids_x_shared[threadIdx.x]);
        atomicAdd(new_centroids_y + threadIdx.x, new_centroids_y_shared[threadIdx.x]);
        atomicAdd(centroid_point_nums + threadIdx.x, centroid_point_nums_shared[threadIdx.x]);
    }
}
```

| Optimization | Benefit |
|:-------------|:--------|
| Shared memory | Reduces global atomic contention |
| Two-level reduction | Block-local first, then global merge |
| Separate count array | Enables mean calculation in next kernel |

### Kernel 3: Finalize and Check Convergence

```cpp
__global__ void finalize_recalculation_and_compare_kernel(
    const int* centroid_point_nums,
    float* new_centroids_x, float* new_centroids_y,
    const float* prev_centroids_x, const float* prev_centroids_y,
    int k, int* num_converged_centroids) 
{
    int idx = threadIdx.x;

    if (idx == 0) *num_converged_centroids = 0;
    __syncthreads();

    if (idx < k) {
        int count = centroid_point_nums[idx];
        if (count > 0) {
            new_centroids_x[idx] /= count;
            new_centroids_y[idx] /= count;
        } else {
            // Keep previous centroid if no points assigned
            new_centroids_x[idx] = prev_centroids_x[idx];
            new_centroids_y[idx] = prev_centroids_y[idx];
        }

        // Check convergence
        float dx = new_centroids_x[idx] - prev_centroids_x[idx];
        float dy = new_centroids_y[idx] - prev_centroids_y[idx];
        if (dx*dx + dy*dy < 1.0e-8f) {
            atomicAdd(num_converged_centroids, 1);
        }
    }
}
```

### Exercise 1: Algorithm Analysis

**Question 1**: In the assignment kernel, what is the time complexity per thread if k = 100?

Your answer: **O(k), so for k = 100 each thread performs 100 distance comparisons. Each comparison computes a squared Euclidean distance and updates the current minimum, so the assignment kernel scales linearly with the number of clusters.**

**Question 2**: Why does the recalculation kernel use shared memory atomics before global atomics?

Your answer: **Shared memory atomics first aggregate points inside each block, which greatly reduces the number of slower global atomic operations and lowers contention on global memory. Instead of every point directly updating global centroid sums, each block produces partial sums locally and only merges k partial results to global memory.**

**Question 3**: What happens if a centroid has no points assigned to it (empty cluster)?

Your answer: **The implementation keeps that centroid at its previous position instead of dividing by zero or moving it to an invalid value. This preserves a valid centroid for the next iteration and avoids NaN values propagating through later distance calculations.**

## 3. Setup: Generate Test Data

In [ ]:
%%bash
# Generate test cases
python3 geninput.py
echo "Test cases generated:"
ls -la testcases/

## 4. Compile the Program

In [ ]:
%%bash
# Compile K-Means program
hipcc -O2 main.hip -o exe_kmeans
echo "Compilation complete."

## 5. Run Test Cases

In [ ]:
%%bash
echo "=== Test 1: Small random case ==="
head -5 testcases/1.in
echo "..."
./exe_kmeans < testcases/1.in

In [ ]:
%%bash
echo "=== Test 3: Minimal case (N=1, k=1) ==="
cat testcases/3.in
echo "Result:"
./exe_kmeans < testcases/3.in

In [ ]:
%%bash
echo "=== Test 4: Large case (N=50000, k=50) ==="
time ./exe_kmeans < testcases/4.in > /dev/null

### Exercise 2: Performance Analysis

Record the execution time for test 4:

- Sample size: 50,000
- Clusters (k): 50
- Max iterations: 100
- Execution time: **0m0.085s**

**Question**: What is the dominant computation in each iteration?

Your answer: **The assignment step dominates: every point computes its squared distance to every centroid, giving O(N * k) distance calculations per iteration. For N = 50,000 and k = 50, this means 2.5 million distance evaluations per iteration. The update step is O(N + k) and mainly performs atomic accumulation and division, so the nearest-centroid search is the main computational bottleneck when k is significant.**

## 6. Parallelization Analysis

### Assignment Phase

| Aspect | Value |
|:-------|:------|
| Parallelism | N threads (one per point) |
| Work per thread | k distance calculations |
| Memory pattern | Coalesced reads |
| Synchronization | None needed |

### Update Phase

| Aspect | Value |
|:-------|:------|
| Parallelism | N threads accumulate, k threads finalize |
| Atomics | 3 per point (shared), 3k per block (global) |
| Synchronization | Block-level sync for shared memory |

### Exercise 3: Compute Intensity

For N = 50,000 points and k = 50 clusters:

**Assignment kernel:**
- Distance calculations per point: **50**
- Total distance calculations: **50,000 * 50 = 2,500,000**

**Update kernel:**
- Shared memory atomics: **3 * 50,000 = 150,000**
- Global atomics (with 256 threads/block): **ceil(50,000 / 256) * 3 * 50 = 196 * 150 = 29,400**

**Analysis:** The assignment kernel performs far more arithmetic than the update kernel. The update kernel still matters because atomics can serialize when many points map to the same centroid, but the shared-memory reduction reduces global atomic pressure from one global update per point to one global merge per centroid per block.

## 7. Memory Access Patterns

### Assignment Kernel

```
Thread 0:  data_x[0], data_y[0], centroids[0..k-1]
Thread 1:  data_x[1], data_y[1], centroids[0..k-1]
Thread 2:  data_x[2], data_y[2], centroids[0..k-1]
...
```

| Access Type | Pattern |
|:------------|:--------|
| Data points | Coalesced (consecutive threads read consecutive elements) |
| Centroids | Broadcast (all threads read same k values) |

### Exercise 4: Memory Optimization

**Question 1**: The centroids are read by every thread. How could you optimize this access pattern?

Your answer: **Load centroids into faster memory that can be reused by the block, such as shared memory, or place small/read-only centroid arrays in constant/read-only cache. This avoids repeatedly fetching the same centroid values from global memory. Because all threads in a block scan the same centroid list, caching or staging centroids improves data reuse and reduces global memory bandwidth demand.**

**Question 2**: If k = 1000, what would be the impact on cache performance?

Your answer: **Each thread would scan many more centroid values, increasing memory traffic and cache pressure. The centroid working set may no longer stay hot in cache, causing more cache misses and making the assignment kernel slower. The current fixed shared-memory arrays of size 100 would also need to be redesigned for k = 1000, for example by using dynamic shared memory, tiling centroids, or processing centroids in batches.**

## 8. Convergence Detection

The algorithm checks if centroids have stopped moving:

```cpp
float dx = new_centroids_x[idx] - prev_centroids_x[idx];
float dy = new_centroids_y[idx] - prev_centroids_y[idx];
if (dx*dx + dy*dy < 1.0e-8f) {
    atomicAdd(num_converged_centroids, 1);
}
```

Early termination occurs when `num_converged_centroids == k`.

### Exercise 5: Convergence Analysis

**Question 1**: What is the convergence threshold in this implementation?

Your answer: **The squared centroid movement threshold is 1.0e-8. A centroid is treated as converged when dx*dx + dy*dy < 1.0e-8. Since the comparison uses squared distance, the code avoids the extra cost of a square root while still measuring whether the centroid movement is very small.**

**Question 2**: Why is it important to check convergence rather than always running max_iterations?

Your answer: **Convergence checking avoids unnecessary iterations after the centroids have stabilized, reducing runtime and saving GPU work while producing the same final clusters. This is especially useful for large datasets because each extra iteration repeats both the O(N * k) assignment step and the atomic update step.**

**Question 3**: How does the double-buffering (swapping prev and new centroids) help?

Your answer: **Double-buffering keeps the previous centroids and the newly computed centroids in separate arrays, so the program can safely compare old and new values and avoid overwriting data that is still needed during the current iteration. Swapping the pointers also avoids copying centroid arrays every iteration, making the iterative loop simpler and more efficient.**

## 9. Limitations and Improvements

### Current Limitations

| Limitation | Impact |
|:-----------|:-------|
| Fixed shared memory size (100) | Maximum k = 100 |
| 2D points only | Cannot handle higher dimensions |
| No k-means++ initialization | May converge to local optima |

### Possible Improvements

1. **Dynamic shared memory**: Support larger k values
2. **Warp shuffle**: Reduce shared memory atomics for small k
3. **Batched distance**: Process multiple points per thread
4. **Hierarchical reduction**: Multi-level centroid accumulation

## 10. Summary

### Key Concepts

| Concept | Description |
|:--------|:------------|
| **K-Means** | Iterative clustering algorithm with assignment and update phases |
| **Parallel Assignment** | Each point independently finds nearest centroid |
| **Reduction for Update** | Atomic accumulation of point coordinates per cluster |
| **Convergence Detection** | Early termination when centroids stop moving |

### GPU Optimization Patterns Used

| Pattern | Application |
|:--------|:------------|
| Shared memory reduction | Centroid accumulation |
| Two-level atomics | Local then global |
| Double buffering | Swap prev/new centroids |
| Coalesced memory access | Data point reads |

### Complexity Analysis

Per iteration:
- Assignment: O(N * k) distance computations
- Update: O(N) accumulations + O(k) divisions
- Total: O(N * k) per iteration

## 11. Challenge Exercises

### Challenge 1: Support Larger k

Modify the kernel to use dynamic shared memory allocation:
```cpp
extern __shared__ float shared_mem[];
```

### Challenge 2: Add Distance Output

Modify the program to output the sum of squared distances (inertia) after clustering.

### Challenge 3: Extend to 3D

Add support for 3D point clustering by adding z coordinates.

### Challenge 4: K-Means++ Initialization

Implement K-Means++ initialization on GPU:
1. Choose first centroid randomly
2. For each subsequent centroid, choose with probability proportional to distance squared

### Challenge 5: Mini-Batch K-Means

Process random subsets of points each iteration to speed up convergence on very large datasets.

---

## Lab Files Reference

| File | Description |
|:-----|:------------|
| `main.hip` | Complete K-Means implementation with 3 kernels |
| `geninput.py` | Test case generator |
| `Makefile` | Build configuration |